# 04 · Table III: which candidates to prune (cache only)

All rules terminate the **same number** of still-running candidates at the
**same checkpoint B = 110** and differ only in *which* ones; survivors are then
selected by the full-solution verifier. Finished candidates are never pruned.

Note on the published table: the *random* and *oracle* rows were originally
measured at B = 126 on the same candidates. This notebook computes **every**
row at B = 110 so the table is internally consistent; use its output for those
two rows.

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gbt] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
from lvpp.caches import load_prefix_C
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier
from lvpp.pruning import prune_keep_and_cost, full_cost, rule_values
from lvpp.stats import paired
PR = PATHS["prefix_root"]; B = 110
teF = load_prefix_C(PR, "test", 169); trF = load_prefix_C(PR, "pool", 169)
sF = fit_verifier(trF.P, trF.Z, trF.y, seed=0).score(teF.P, teF.Z)
h_maj = selection_hits(teF, "majority"); h_ver = selection_hits(teF, "weighted", scores=sF)
c_full = full_cost(teF.extra["n_total"])
teB = load_prefix_C(PR, "test", B); trB = load_prefix_C(PR, "pool", B)
sB = fit_verifier(trB.P, trB.Z, trB.y, seed=0).score(teB.P, teB.Z)
assert teB.extra["prefix_lp"] is not None, "prefix log-probs missing: recapture with notebook 01"
print(f"majority {100*h_maj.mean():.2f}% | unpruned verifier {100*h_ver.mean():.2f}% (paper 73.92 / 77.03)")
print(f"finished before B={B}: {100*teB.extra['finished'].mean():.1f}%")

In [ ]:
def run(values, rho):
    keep, cost = prune_keep_and_cost(values, teB.extra["finished"], teF.extra["n_total"], B, rho)
    h = selection_hits(teF, "weighted", scores=sF, keep=keep)
    return h, 100 * (1 - cost / c_full)

rows = []
def add(rule, rho, h, saved):
    pv = paired(h, h_ver)
    rows.append(dict(rule=rule, rho=rho, acc=round(100 * h.mean(), 2), saved=round(saved, 1),
                     vs_unpruned_verifier=round(pv["diff"], 2), p=pv["p"]))

nt = teF.extra["n_total"]
for rho in [0.5]:
    add("keep the longest prefixes", rho, *run(rule_values("longest", n_total=nt), rho))
    hs, ss = zip(*[run(rule_values("random", n_total=nt, rng=np.random.default_rng(i)), rho) for i in range(20)])
    accs = [100 * h.mean() for h in hs]
    rows.append(dict(rule="random (mean of 20)", rho=rho, acc=round(float(np.mean(accs)), 2),
                     saved=round(float(np.mean(ss)), 1), vs_unpruned_verifier=None, p=None))
    print(f"random: {np.mean(accs):.2f} +/- {np.std(accs):.2f}")
    add("oracle (abort incorrect first)", rho, *run(rule_values("oracle", labels=teB.y.numpy()), rho))
for rho in [0.125, 0.25, 0.375, 0.5]:
    add("token confidence (prefix log-prob)", rho, *run(teB.extra["prefix_lp"], rho))
    add("LVPP (prefix verifier score)", rho, *run(sB, rho))
T3 = pd.DataFrame(rows); print(T3.to_string(index=False))
T3.to_csv(f"{RESULTS}/table3.csv", index=False)

In [ ]:
EXPECTED = {"LVPP 0.125": 77.10, "LVPP 0.25": 76.88, "LVPP 0.375": 76.57, "LVPP 0.5 (seed 0)": 76.27,
            "conf 0.125": 77.10, "conf 0.25": 76.50, "conf 0.375": 75.44, "conf 0.5": 75.06,
            "longest 0.5": 74.30}
got = {f"LVPP {r}": a for r, a in zip([0.125, 0.25, 0.375], T3[T3.rule.str.startswith("LVPP")].acc[:3])}
got["LVPP 0.5 (seed 0)"] = float(T3[(T3.rule.str.startswith("LVPP")) & (T3.rho == 0.5)].acc.iloc[0])
for r in [0.125, 0.25, 0.375, 0.5]:
    got[f"conf {r}"] = float(T3[(T3.rule.str.startswith("token")) & (T3.rho == r)].acc.iloc[0])
got["longest 0.5"] = float(T3[T3.rule.str.startswith("keep")].acc.iloc[0])
report(got, EXPECTED, list(EXPECTED))
print("\nThe paper's rho = 0.5 LVPP row (76.55, 16.3%) is the 3-seed mean from notebook 03.")